<a href="https://colab.research.google.com/github/Irene-sketch/AI_ML_sem5/blob/main/Stackingclassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

# 1. Generate some synthetic data for classification
X, y = make_classification(n_samples=1000, n_features=20,
                           n_informative=10, n_redundant=5,
                           random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

print("Data generated successfully.")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

Data generated successfully.
X_train shape: (750, 20)
y_train shape: (750,)


Now, let's define the base models and the meta-model for our stacking classifier.

In [ ]:
# 2. Define base models
# For this example, we'll use a single RandomForestClassifier as a 'base model'
# In a real scenario, you might use multiple diverse base models.
base_models = [
    ('rf1', RandomForestClassifier(n_estimators=100, random_state=42))
    # You could add more diverse base models here, e.g., other tree-based models, logistic regression, etc.
    # ('lr', LogisticRegression(solver='liblinear', random_state=42))
]

# 3. Define the meta-model (final estimator)
meta_model = SVC(kernel='linear', random_state=42, probability=True)

print("Base model and meta-model defined.")

Base model and meta-model defined.


Next, we'll create and train the `StackingClassifier`.

In [ ]:
# 4. Create the StackingClassifier
# The 'stack_method' specifies how predictions from base models are combined for the meta-model.
# 'auto' tries to use 'predict_proba' if available, otherwise 'decision_function', then 'predict'.
stacking_classifier = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5, # Number of cross-validation folds to use for fitting the base models
    stack_method='auto', # How to combine predictions for the meta-model
    n_jobs=-1, # Use all available CPU cores
    verbose=1 # Print progress messages
)

# 5. Train the stacking classifier
print("\nTraining StackingClassifier...")
stacking_classifier.fit(X_train, y_train)
print("StackingClassifier training complete.")


Training StackingClassifier...
StackingClassifier training complete.


Finally, let's make predictions and evaluate the performance of our stacking model.

In [ ]:
# 6. Make predictions on the test set
y_pred = stacking_classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy of the StackingClassifier: {accuracy:.4f}")

# You can also inspect the performance of individual base estimators
for name, estimator in stacking_classifier.named_estimators_.items():
    # Note: named_estimators_ only contains the estimators fitted on the full training data,
    # not the ones used for generating meta-features in cross-validation.
    # For a direct comparison, you'd usually train them separately on X_train.
    base_pred = estimator.predict(X_test)
    base_accuracy = accuracy_score(y_test, base_pred)
    print(f"Accuracy of {name} (base estimator): {base_accuracy:.4f}")


Accuracy of the StackingClassifier: 0.9160
Accuracy of rf1 (base estimator): 0.9200
